# exam-diag demo

输入一批作答记录，输出**根因概念**（最该补的上游知识点）+ 补习路径 + 7 天复习计划。

直接 `jupyter lab` / `VS Code` 打开本笔记本从头跑到尾即可，无需联网。

In [1]:
from examdiag.diagnose import diagnose
from examdiag.schema import load_attempts, load_concepts, load_questions
from examdiag.srs import plan_week

concepts = load_concepts('../data/concepts/408.yaml')
questions = load_questions('../data/questions/408.jsonl')
attempts = load_attempts('../data/demo_attempts.jsonl')
print(f'attempts={len(attempts)} questions={len(questions)} '
      f'concepts={len(concepts.concepts)}')

attempts=500 questions=150 concepts=60


## 1. 根因诊断

对每个概念拟合 BKT（EM）得到掌握度，错题沿前置图向上游回溯：
前置也未掌握的概念是症状，最上游的破损概念才是根因。

In [2]:
diag = diagnose(attempts, questions, concepts)

for i, r in enumerate(diag.root_causes, 1):
    print(f'#{i} {r.name} ({r.concept_id})')
    print(f'   掌握度 {r.mastery:.2f} | 错题 {r.wrong_count} | 距离 {r.min_distance}')
    print('   补习路径:', ' → '.join(concepts.concepts[c].name for c in r.remedy_path))
    explained = '、'.join(concepts.concepts[c].name for c in r.explains)
    print('   解释症状:', explained or '—')
    print()

#1 二叉树遍历 (ds.tree.traverse)
   掌握度 0.10 | 错题 26 | 距离 0
   补习路径: 二叉树遍历 → 二叉搜索树 → AVL平衡旋转 → AVL删除与综合调整
   解释症状: 二叉搜索树、AVL平衡旋转、AVL删除与综合调整、树与森林的遍历、B树与B+树、线索二叉树



## 2. 未来 7 天复习计划

间隔与掌握度成反比：掌握度越低，出现越频繁。

In [3]:
for p in plan_week(diag):
    names = [concepts.concepts[c].name for c in p.concepts]
    print(f'Day {p.day}:', '、'.join(names) if names else '（休息）')

Day 1: 二叉树遍历
Day 2: 二叉树遍历
Day 3: 二叉树遍历
Day 4: 二叉树遍历
Day 5: 二叉树遍历
Day 6: 二叉树遍历
Day 7: 二叉树遍历


## 3. 全部概念掌握度一览（诊断涉及到的）

In [4]:
rows = sorted(diag.mastery.items(), key=lambda kv: kv[1])
for cid, m in rows[:8]:
    bar = '█' * round(m * 30)
    print(f'{concepts.concepts[cid].name:12s} {m:5.2f} |{bar:<30}| {cid}')
print(f'... 共 {len(rows)} 个有数据概念')

AVL平衡旋转       0.04 |█                             | ds.bst.rotate
AVL删除与综合调整    0.04 |█                             | ds.bst.delete
二叉树遍历         0.10 |███                           | ds.tree.traverse
树与森林的遍历       0.54 |████████████████              | ds.forest.traverse
二叉搜索树         0.90 |███████████████████████████   | ds.bst.bst
循环队列          0.95 |████████████████████████████  | ds.queue.circular
双链表           0.99 |██████████████████████████████| ds.linked.dlist
二叉树定义与性质      1.00 |██████████████████████████████| ds.tree.binary
... 共 60 个有数据概念


## 4. 更多

- `viz/report.html`：浏览器打开，依赖图 + 掌握度着色 + 点击查看涉及错题
- `examdiag diagnose --attempts ... --concepts ... --plan --json`：命令行 / JSON 输出
- `docs/DATA_FORMAT.md`：三类专业文件的格式说明